# PVision AI: Photovoltaic Fault Dataset Inspection
**Project:** A Hybrid CNN-ANN Framework for Photovoltaic Fault Detection and Power Generation Prediction  
**Phase 1:** Dataset Inspection and Exploratory Analysis (Images: GASF and I-V Curves)  

This notebook inspects the PV fault image dataset: image file verification, class counts, missing files, corrupted images, duplicate detection, and visual sample exploration.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import json
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

from src.utilities.config import config
from src.data.inspect_pv_fault import PVFaultInspector

print(f"Project Root: {config.PROJECT_ROOT}")
print(f"PV Fault Data Path: {config.PV_FAULT_DATA_DIR}")
print(f"Classes: {config.PV_FAULT_CLASSES}")

## 1. Execute Dataset Inspection
Run `PVFaultInspector` to scan image files, detect missing sequence indices, check corruptions, and compute sample counts.

In [ ]:
inspector = PVFaultInspector()
# Fast duplicate check on a subset for interactive speed
summary = inspector.inspect(check_duplicates=True, max_duplicate_samples=2500)

## 2. Class Distribution Summary Table

In [ ]:
for modality, mod_rep in summary.get("modality_reports", {}).items():
    print(f"=== Modality: {modality} (Total: {mod_rep['total_images']:,} images) ===")
    df_dist = pd.DataFrame({
        "Count": mod_rep["class_counts"],
        "Percentage (%)": mod_rep["class_distribution_percentage"]
    })
    display(df_dist)
    
    if mod_rep.get("missing_sequence_details"):
        print("\nMissing Sequence Details:")
        for cls_name, info in mod_rep["missing_sequence_details"].items():
            print(f"  - {cls_name}: {info['missing_count']} missing files! Missing ranges: {info['missing_index_ranges']}")
    print("\n" + "-" * 50)

## 3. Visual Exploration: Sample Images Across Classes and Modalities

In [ ]:
classes = config.PV_FAULT_CLASSES
modalities = ["gasf_images", "iv_images"]

fig, axes = plt.subplots(len(classes), len(modalities), figsize=(8, 2.5 * len(classes)))

for r, cls_name in enumerate(classes):
    for c, mod in enumerate(modalities):
        cls_dir = config.PV_FAULT_DATA_DIR / mod / cls_name
        ax = axes[r, c]
        if cls_dir.is_dir():
            files = list(cls_dir.glob("*.png"))
            if files:
                img = Image.open(files[0])
                ax.imshow(img)
                ax.set_title(f"{mod}: {cls_name}\nSize: {img.size}", fontsize=9)
        ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Key Findings & Preprocessing Strategy for VS Code Manual Training
1. **GASF Modality:** All 7 classes have exactly 5,000 samples (35,000 total images). Images are 256x256 RGB. Balanced distribution.
2. **I-V Modality:** 6 classes have 5,000 samples, while `short_circuit` has 4,484 samples (516 missing sequence files: 1-513, 597-599). Images are 1343x808 RGB.
3. **CNN Preprocessing Recommendation:** Apply standard resize to 224x224, random horizontal/vertical flips, and ImageNet channel normalization.